In [0]:
%run "./Data transformations and training"

In [0]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import StandardScaler

In [0]:
# Fill missing values and convert types
for col in ["lag_1", "lag_7", "rolling_avg_7"]:
    X_train[col] = X_train[col].fillna(0).astype(float)
    X_test[col] = X_test[col].fillna(0).astype(float)

In [0]:
# Scale features for better RNN performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [0]:
# Reshape data for RNN (samples, timesteps, features)
# For RNN, we need 3D input: (batch_size, timesteps, features)
X_train_rnn = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_rnn = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

print(f"Training data shape: {X_train_rnn.shape}")
print(f"Test data shape: {X_test_rnn.shape}")

In [0]:
# Build LSTM model
model = Sequential([
    LSTM(64, activation='relu', input_shape=(1, X_train_scaled.shape[1]), return_sequences=True),
    Dropout(0.2),
    LSTM(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['mae']
)

model.summary()

In [0]:
# Train the model
history = model.fit(
    X_train_rnn,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

In [0]:
# Make predictions
predictions = model.predict(X_test_rnn).flatten()

In [0]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("RNN MAE:", mae)
print("RNN RMSE:", rmse)

In [0]:
import shap

# Use DeepExplainer for neural networks
# Select a background sample for SHAP
background = X_train_rnn[:100]
explainer = shap.DeepExplainer(model, background)

# Calculate SHAP values for test set (use a sample for speed)
test_sample = X_test_rnn[:100]
shap_values = explainer.shap_values(test_sample)

In [0]:
# Reshape SHAP values for visualization
# SHAP values need to be 2D for summary plot
shap_values_2d = shap_values[0].reshape(test_sample.shape[0], -1)
test_sample_2d = test_sample.reshape(test_sample.shape[0], -1)

# Create feature names for better visualization
feature_names = ["day_of_week", "month", "week_of_year", "lag_1", "lag_7", "rolling_avg_7"]

shap.summary_plot(shap_values_2d, test_sample_2d, feature_names=feature_names)

In [0]:
# Waterfall plot for first prediction
shap.plots.waterfall(shap.Explanation(values=shap_values_2d[0], 
                                       base_values=explainer.expected_value[0], 
                                       data=test_sample_2d[0],
                                       feature_names=feature_names))

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

plt.plot(y_test.values, label="Actual")
plt.plot(predictions, label="RNN Predicted")

plt.legend()
plt.title("Demand Forecasting - RNN Model")
plt.xlabel("Sample")
plt.ylabel("Quantity")

plt.show()

In [0]:
# Plot training history
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['mae'], label='Training MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.title('Model MAE')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()

plt.tight_layout()
plt.show()

In [0]:
import pandas as pd

# Create results dataframe
rnn_results = pd.DataFrame({
    "Actual": y_test.values,
    "RNN_Prediction": predictions
})

rnn_results.head()

In [0]:
# Convert to Spark DataFrame and display
spark_rnn_results = spark.createDataFrame(rnn_results)
display(spark_rnn_results)

In [0]:
# Save results to table
spark.sql('DROP TABLE IF EXISTS workspace.general_use.rnn_demand_forecast_results')
spark_rnn_results.write.mode("overwrite").saveAsTable("workspace.general_use.rnn_demand_forecast_results")